# Notebook 06 — Visualizing Your Designed Binders

**Learning objectives:**
- Load designed binder PDB files and visualize them in 3D
- Check that the binder sits over the hotspot residues
- Compare the binder interface in different representation modes
- Generate publication-quality screenshots for your Design Report

**Time:** ~45 minutes

**Companion wiki pages:** [6.4 Visualization](https://rucha1796.github.io/internship-bindcraft-2026/m6_04_visualization/) | [7.3 Good vs. Bad](https://rucha1796.github.io/internship-bindcraft-2026/m7_03_good_vs_bad/)

---
> No GPU required. Works with the pregenerated PDL1_8aok dataset PDB files.

In [ ]:
!pip install py3Dmol requests -q
import py3Dmol, requests, glob, os
import pandas as pd
print("✓ Ready")

## Cell 1 — Load dataset and find PDB files

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

DATA_DIR = "/content/drive/MyDrive/bindcraft/PDL1_8aok"

# Find ranked PDB files
ranked_pdbs = sorted(glob.glob(f"{DATA_DIR}/Accepted/Ranked/*.pdb"))

# If pregenerated PDBs not on Drive, download 8AOK as a stand-in
if not ranked_pdbs:
    print("No PDB files found in pregenerated dataset.")
    print("Downloading 8AOK (PD-L1 + designed binder) from RCSB as demonstration...")
    pdb_8aok = requests.get("https://files.rcsb.org/download/8AOK.pdb").text
    os.makedirs("/content/demo_designs", exist_ok=True)
    with open("/content/demo_designs/1_design.pdb", "w") as f:
        f.write(pdb_8aok)
    ranked_pdbs = ["/content/demo_designs/1_design.pdb"]
    print("✓ Using 8AOK as demonstration (this IS a real designed binder on PD-L1)")

print(f"\nFound {len(ranked_pdbs)} PDB file(s):")
for p in ranked_pdbs[:5]:
    print(f"  {os.path.basename(p)}")

# Load the CSV for metric reference
csv_path = f"{DATA_DIR}/final_design_stats.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path).sort_values("Average_i_pTM", ascending=False).reset_index(drop=True)
    print(f"\nCSV loaded: {len(df)} designs")
else:
    df = None
    print("\nCSV not available — metrics will not be shown alongside structures")

## Cell 2 — The standard view function

We define a reusable visualization function used throughout this notebook.

In [ ]:
HOTSPOT_RESIDUES = [54, 56, 111, 115, 123]

def visualize_complex(
    pdb_path,
    target_chain="A",
    binder_chain="B",
    target_color="#3c5b6f",
    binder_color="#B76E79",
    hotspots=HOTSPOT_RESIDUES,
    surface_opacity=0.0,
    width=750,
    height=500,
    label=""
):
    """Standard visualization for a designed binder complex."""
    with open(pdb_path) as f:
        pdb_str = f.read()

    view = py3Dmol.view(width=width, height=height)
    view.addModel(pdb_str, "pdb")

    # Target chain
    view.setStyle({"chain": target_chain}, {"cartoon": {"color": target_color}})

    # Binder chain
    view.setStyle({"chain": binder_chain}, {"cartoon": {"color": binder_color}})

    # Semi-transparent surface on target (optional)
    if surface_opacity > 0:
        view.addSurface(
            py3Dmol.VDW,
            {"opacity": surface_opacity, "color": target_color},
            {"chain": target_chain}
        )

    # Hotspot residues as yellow spheres
    for resnum in hotspots:
        view.addStyle(
            {"chain": target_chain, "resi": str(resnum)},
            {"sphere": {"color": "yellow", "radius": 0.9}}
        )

    view.zoomTo()
    view.setBackgroundColor("white")
    if label:
        print(f"\n--- {label} ---")
        print(f"Blue = PD-L1 (chain {target_chain}) | Rose = binder (chain {binder_chain}) | Yellow = hotspots")
    view.show()
    return view

print("✓ Visualization function defined")

## Cell 3 — Visualize the top design (cartoon)

In [ ]:
top_pdb = ranked_pdbs[0]

if df is not None and len(df) > 0:
    best = df.iloc[0]
    print(f"Best design: {best['binder_name']}")
    print(f"  i_pTM:               {best.get('Average_i_pTM', 'N/A'):.3f}")
    print(f"  ShapeComplementarity: {best.get('Average_ShapeComplementarity', 'N/A'):.3f}")
    print(f"  Binder RMSD:          {best.get('Average_Binder_RMSD', 'N/A'):.2f} Å")
    print(f"  dG:                   {best.get('Average_dG', 'N/A'):.1f} kcal/mol")

visualize_complex(top_pdb, label="Rank 1 — Cartoon view")

## Cell 4 — With surface (see the interface)

In [ ]:
visualize_complex(
    top_pdb,
    surface_opacity=0.35,
    label="Rank 1 — With PD-L1 surface (opacity 0.35)"
)
print("Does the rose binder sit inside or on top of the blue surface?")

## Cell 5 — View all top designs (gallery)

In [ ]:
n_designs = min(5, len(ranked_pdbs))

for i, pdb_path in enumerate(ranked_pdbs[:n_designs]):
    rank = i + 1
    name = os.path.basename(pdb_path).replace(".pdb", "")
    
    if df is not None and i < len(df):
        row = df.iloc[i]
        label = (f"Rank {rank}: {row.get('binder_name', name)} "
                 f"| i_pTM={row.get('Average_i_pTM', 0):.3f} "
                 f"| SC={row.get('Average_ShapeComplementarity', 0):.3f} "
                 f"| RMSD={row.get('Average_Binder_RMSD', 0):.2f}Å")
    else:
        label = f"Rank {rank}: {name}"
    
    visualize_complex(pdb_path, width=700, height=420, label=label)

## Cell 6 — Visual inspection worksheet

For each of your top 5 designs, fill in this table by observing the 3D structures above.

In [ ]:
import pandas as pd

# Fill in these values after looking at each structure
# Y = yes, N = no, P = partial

visual_worksheet = [
    # (rank, binder_sits_on_hotspots, interface_tight, binder_is_helical, notes)
    (1, "?", "?", "?", "Edit this cell with your observation"),
    (2, "?", "?", "?", "Edit this cell with your observation"),
    (3, "?", "?", "?", "Edit this cell with your observation"),
    (4, "?", "?", "?", "Edit this cell with your observation"),
    (5, "?", "?", "?", "Edit this cell with your observation"),
]

worksheet_df = pd.DataFrame(
    visual_worksheet,
    columns=["Rank", "Covers hotspots?", "Interface tight?", "Binder helical?", "Notes"]
)
worksheet_df.set_index("Rank", inplace=True)

print("Visual Inspection Worksheet")
print("Instructions: Run Cell 5, then edit 'visual_worksheet' above based on what you see")
print()
print(worksheet_df.to_string())

## Cell 7 — Compare best vs. worst accepted design

In [ ]:
if len(ranked_pdbs) >= 2:
    best_pdb = ranked_pdbs[0]
    worst_pdb = ranked_pdbs[-1]  # Lowest i_pTM among accepted
    
    print("BEST design (highest i_pTM):")
    if df is not None and len(df) > 0:
        best_row = df.iloc[0]
        print(f"  i_pTM = {best_row.get('Average_i_pTM', 'N/A'):.3f}")
        print(f"  ShapeComp = {best_row.get('Average_ShapeComplementarity', 'N/A'):.3f}")
    visualize_complex(best_pdb, width=700, height=420, label="BEST design")
    
    print("\nLOWEST-ranked accepted design (still passed all filters):")
    if df is not None and len(df) > 1:
        worst_row = df.iloc[-1]
        print(f"  i_pTM = {worst_row.get('Average_i_pTM', 'N/A'):.3f}")
        print(f"  ShapeComp = {worst_row.get('Average_ShapeComplementarity', 'N/A'):.3f}")
    visualize_complex(worst_pdb, width=700, height=420, label="LOWEST accepted design")
    
    print("\nQ: Can you see visual differences between the best and lowest designs?")
    print("Q: Does the lowest design cover the hotspot residues as well as the best?")
else:
    print("Only one design available — run with more trajectories to get a comparison range.")

## Cell 8 — Save visualization notes

Run this to print a clean summary for your Design Recommendation Report.

In [ ]:
print("=" * 60)
print("VISUALIZATION SUMMARY FOR DESIGN REPORT")
print("=" * 60)

if df is not None:
    for i, (pdb_path) in enumerate(ranked_pdbs[:5]):
        rank = i + 1
        if i < len(df):
            row = df.iloc[i]
            print(f"\nRank {rank}: {row.get('binder_name', os.path.basename(pdb_path))}")
            print(f"  PDB file:     {os.path.basename(pdb_path)}")
            print(f"  i_pTM:        {row.get('Average_i_pTM', 'N/A'):.3f}")
            print(f"  ShapeComp:    {row.get('Average_ShapeComplementarity', 'N/A'):.3f}")
            print(f"  Binder RMSD:  {row.get('Average_Binder_RMSD', 'N/A'):.2f} Å")
            print(f"  dG:           {row.get('Average_dG', 'N/A'):.1f} kcal/mol")
            print(f"  H-bonds:      {row.get('Average_n_InterfaceHbonds', 'N/A'):.0f}")
            print(f"  Visual notes: [fill in from Cell 6 worksheet]")
else:
    print("\nLoad CSV in Cell 1 to see metrics alongside file names")

print("\n" + "=" * 60)
print("Next step: Copy this summary into your Design Recommendation Report")
print("Then add screenshots from Cell 3 or Cell 5")
print("(Use your OS screenshot tool: Cmd+Shift+4 on Mac, Win+Shift+S on Windows)")

---
**Next:** Notebook 07 — Failure analysis: understanding what didn't work and why